# API Testing Notebook

## Service Ports
- **Auth Service:** 8081
- **API Gateway:** 8084
- **User Profile:** 8083
- **Notes:** 8088

---

## Step 1: Create user1 via Auth Service (Direct)

**Endpoint:** `POST http://localhost:8081/auth/signup`

**Status:** ✅ SUCCESS

In [ ]:
import requests
import json

# Step 1: Create user1 via Auth Service directly
response = requests.post(
    "http://localhost:8081/auth/signup",
    headers={"Content-Type": "application/json"},
    json={
        "tenantId": "11111111-1111-1111-1111-111111111111",
        "email": "user1@example.com",
        "password": "pass1",
        "name": "user1",
        "joinMethod": "SELF_SIGNUP"
    }
)

print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

### Result (user1)
```json
{
  "accessToken": "cd6e3ab2-5f30-414e-b219-b369a6ce8c94",
  "expiresIn": 3600,
  "refreshToken": "d8387ddd-4f2a-4517-8194-c570c48d4d8b",
  "tenantId": "11111111-1111-1111-1111-111111111111",
  "userId": "cd534d0d-1cc3-46fa-9a4b-e34ec4f9ca20"
}
```

---

## Step 2: Create user2 via API Gateway

**Endpoint:** `POST http://localhost:8084/v1/auth/signup`

**Note:** API Gateway uses `/v1/` prefix for all routes

**Status:** ✅ SUCCESS

In [2]:
# delete user2 from first
response = requests.delete(
    "http://localhost:8084/v1/auth/user2",
    headers={"Authorization": f"Bearer {access_token}"}
)

# print response success or failure
if response.status_code == 204:
    print("User2 deleted successfully.")
else:
    print("Failed to delete User2.")

# Step 2: Create user2 via API Gateway
response = requests.post(
    "http://localhost:8084/v1/auth/signup",
    headers={"Content-Type": "application/json"},
    json={
        "tenantId": "11111111-1111-1111-1111-111111111111",
        "email": "user2@example.com",
        "password": "pass2",
        "name": "user2",
        "joinMethod": "SELF_SIGNUP"
    }
)

print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

NameError: name 'requests' is not defined

### Result (user2)
```json
{
  "accessToken": "f8cbc88d-43c3-4fde-8247-0acb4dfd71e4",
  "expiresIn": 3600,
  "refreshToken": "9d495cd0-7ab7-4c3f-9c46-181babab484c",
  "tenantId": "11111111-1111-1111-1111-111111111111",
  "userId": "9a18b053-7913-4e4b-b14e-744cc4f3b99a"
}
```

---

## Users Summary

| User | Email | Password | userId | Created Via |
|------|-------|----------|--------|-------------|
| user1 | user1@example.com | pass1 | cd534d0d-1cc3-46fa-9a4b-e34ec4f9ca20 | Auth Service (8081) |
| user2 | user2@example.com | pass2 | 9a18b053-7913-4e4b-b14e-744cc4f3b99a | API Gateway (8084) |

In [ ]:
import requests
import json

# Fetch all tenants from tenant service (port 8082)
response = requests.get("http://localhost:8082/v1/tenants")

print(f"Status: {response.status_code}")
if response.status_code == 200:
    tenants = response.json()
    print(f"\nFound {len(tenants)} tenant(s):\n")
    print(json.dumps(tenants, indent=2))
else:
    print(f"Error: {response.text}")

In [ ]:
import requests
import json

# Fetch all tenants through API Gateway (port 8084)
response = requests.get("http://localhost:8084/v1/tenants")

print(f"Status: {response.status_code}")
if response.status_code == 200:
    tenants = response.json()
    print(f"\nFound {len(tenants)} tenant(s):\n")
    print(json.dumps(tenants, indent=2))
else:
    print(f"Error: {response.text}")

---

## Role-Permission Service (Port 8080)

### List All Permissions

In [ ]:
import requests
import json

# Fetch all permissions from role-permission service (port 8080)
response = requests.get("http://localhost:8080/permissions")

print(f"Status: {response.status_code}")
if response.status_code == 200:
    permissions = response.json()
    print(f"\nFound {len(permissions)} permission(s):\n")
    print(json.dumps(permissions, indent=2))
else:
    print(f"Error: {response.text}")

### List All Roles for Tenant

In [ ]:
import requests
import json

# Tenant ID from previous tests
tenant_id = "11111111-1111-1111-1111-111111111111"

# Fetch all roles for the tenant from role-permission service (port 8080)
response = requests.get(f"http://localhost:8080/tenants/{tenant_id}/roles")

print(f"Status: {response.status_code}")
if response.status_code == 200:
    roles = response.json()
    print(f"\nFound {len(roles)} role(s):\n")
    print(json.dumps(roles, indent=2))
else:
    print(f"Error: {response.text}")

### List All Permission Scopes

In [ ]:
import requests
import json

# Fetch all permission scopes from role-permission service (port 8080)
response = requests.get("http://localhost:8080/permission-scopes")

print(f"Status: {response.status_code}")
if response.status_code == 200:
    scopes = response.json()
    print(f"\nFound {len(scopes)} permission scope(s):\n")
    print(json.dumps(scopes, indent=2))
else:
    print(f"Error: {response.text}")

---

## Create Admin Role with Notes Permissions

**Note:** Role-permission-service uses numeric tenant IDs (Long), not UUIDs.

### Step 1: Create the Notes Permissions

In [ ]:
import requests
import json

# Create notes permissions
# Note: If 'name' is not provided, code is auto-generated as RESOURCE:ACTION
# If 'name' is provided, it becomes the permission code

permissions_to_create = [
    {"description": "Allow creating new notes", "resource": "NOTE", "action": "CREATE", "active": True},
    {"description": "Allow reading notes", "resource": "NOTE", "action": "READ", "active": True},
    {"description": "Allow updating notes", "resource": "NOTE", "action": "UPDATE", "active": True},
    {"description": "Allow deleting notes", "resource": "NOTE", "action": "DELETE", "active": True},
]

created_permissions = []

print("Creating Notes Permissions...")
print("-" * 40)

for perm in permissions_to_create:
    response = requests.post(
        "http://localhost:8080/permissions",
        headers={"Content-Type": "application/json"},
        json=perm
    )
    code = f"{perm['resource']}:{perm['action']}"
    print(f"{code} - Status: {response.status_code}")
    
    if response.status_code in [200, 201]:
        created_permissions.append(response.json())
    else:
        print(f"  Error: {response.text}")

print("-" * 40)
print(f"Created {len(created_permissions)} permission(s)")

### Step 2: Create Admin Role

In [ ]:
import requests
import json

# Role-permission service uses numeric tenant IDs (Long), not UUIDs
# Use tenant ID 1 (or adjust based on your tenant data)
tenant_id = 1

# Create Admin role
response = requests.post(
    f"http://localhost:8080/tenants/{tenant_id}/roles",
    headers={"Content-Type": "application/json"},
    json={
        "name": "admin",
        "description": "Administrator role with full notes permissions",
        "active": True
    }
)

print("Creating Admin role...")
print(f"Status: {response.status_code}")
if response.status_code in [200, 201]:
    admin_role = response.json()
    admin_role_id = admin_role.get("id")
    print(json.dumps(admin_role, indent=2))
    print(f"\nAdmin Role ID: {admin_role_id}")
else:
    print(f"Response: {response.text}")

### Step 3: Assign Notes Permissions to Admin Role

In [ ]:
import requests
import json

# Role-permission service uses numeric tenant IDs (Long)
tenant_id = 1

# First, get the admin role ID (in case you're running this cell separately)
response = requests.get(f"http://localhost:8080/tenants/{tenant_id}/roles")
roles = response.json()
admin_role_id = None
for role in roles:
    if role.get("name") == "admin":
        admin_role_id = role.get("id")
        break

if not admin_role_id:
    print("Admin role not found! Please run the previous cell first.")
else:
    print(f"Found Admin role with ID: {admin_role_id}")
    
    # Assign all notes permissions to the admin role
    notes_permissions = ["NOTE:CREATE", "NOTE:READ", "NOTE:UPDATE", "NOTE:DELETE"]
    
    for perm_code in notes_permissions:
        response = requests.post(
            f"http://localhost:8080/tenants/{tenant_id}/roles/{admin_role_id}/grants",
            headers={"Content-Type": "application/json"},
            json={
                "permissionCode": perm_code,
                "scopeCode": "TENANT"  # Full tenant-wide access for admin
            }
        )
        print(f"Granting {perm_code} to admin - Status: {response.status_code}")
        if response.status_code not in [200, 201]:
            print(f"  Response: {response.text}")

### Step 4: Verify Admin Role Permissions

In [ ]:
import requests
import json

# Role-permission service uses numeric tenant IDs (Long)
tenant_id = 1

# Get admin role ID
response = requests.get(f"http://localhost:8080/tenants/{tenant_id}/roles")
roles = response.json()
admin_role_id = None
for role in roles:
    if role.get("name") == "admin":
        admin_role_id = role.get("id")
        break

if admin_role_id:
    # Fetch all grants for admin role
    response = requests.get(f"http://localhost:8080/tenants/{tenant_id}/roles/{admin_role_id}/grants")
    
    print(f"Permissions granted to Admin role (ID: {admin_role_id}):")
    print(f"Status: {response.status_code}")
    if response.status_code == 200:
        grants = response.json()
        print(f"\nFound {len(grants)} permission grant(s):\n")
        print(json.dumps(grants, indent=2))
    else:
        print(f"Error: {response.text}")
else:
    print("Admin role not found!")

---

## Cleanup: Delete Admin Role and Grants

Run these cells to clean up the test data.

### Delete Permission Grants from Admin Role

In [ ]:
import requests
import json

tenant_id = 1

# Get admin role ID
response = requests.get(f"http://localhost:8080/tenants/{tenant_id}/roles")
if response.status_code != 200:
    print(f"Failed to fetch roles: {response.text}")
else:
    roles = response.json()
    admin_role_id = None
    for role in roles:
        if role.get("name") == "admin":
            admin_role_id = role.get("id")
            break
    
    if not admin_role_id:
        print("Admin role not found - nothing to delete.")
    else:
        print(f"Found Admin role with ID: {admin_role_id}")
        
        # Get all grants for admin role
        response = requests.get(f"http://localhost:8080/tenants/{tenant_id}/roles/{admin_role_id}/grants")
        if response.status_code == 200:
            grants = response.json()
            print(f"Found {len(grants)} grant(s) to delete...")
            
            for grant in grants:
                grant_id = grant.get("id")
                perm_code = grant.get("permissionCode")
                del_response = requests.delete(
                    f"http://localhost:8080/tenants/{tenant_id}/roles/{admin_role_id}/grants/{grant_id}"
                )
                print(f"  Deleting grant {grant_id} ({perm_code}) - Status: {del_response.status_code}")
        else:
            print(f"Failed to fetch grants: {response.text}")

### Delete Admin Role

In [ ]:
import requests
import json

tenant_id = 1

# Get admin role ID
response = requests.get(f"http://localhost:8080/tenants/{tenant_id}/roles")
if response.status_code != 200:
    print(f"Failed to fetch roles: {response.text}")
else:
    roles = response.json()
    admin_role_id = None
    for role in roles:
        if role.get("name") == "admin":
            admin_role_id = role.get("id")
            break
    
    if not admin_role_id:
        print("Admin role not found - nothing to delete.")
    else:
        print(f"Deleting Admin role (ID: {admin_role_id})...")
        response = requests.delete(f"http://localhost:8080/tenants/{tenant_id}/roles/{admin_role_id}")
        
        if response.status_code in [200, 204]:
            print("Admin role deleted successfully!")
        else:
            print(f"Failed to delete: Status {response.status_code}")
            print(f"Response: {response.text}")

### Note on Permissions

Permissions (NOTE:CREATE, NOTE:READ, etc.) are **global** and cannot be deleted via API. They can be marked as deprecated by setting `active: false` via PUT `/permissions/{permissionId}`. The permissions created here will remain in the system for reuse.